# 06 — Pipeline de Entrenamiento Industrial

**Fase 6 · LLM Lab (Etapa 1)**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastian-tf/llm-lab/blob/main/notebooks/06_entrenamiento.ipynb)

## Objetivos de aprendizaje

1. Entender y derivar **next-token prediction** con teacher forcing.
2. Derivar **cross-entropy** desde máxima verosimilitud y calcular perplexity.
3. Construir un **Dataset/DataLoader** con *packing* (sin padding).
4. Implementar el **training loop desde cero**: `train_step`, `eval_step`, checkpointing.
5. Comprender el ruido del gradiente en función del **batch size** y el *critical batch size*.
6. Implementar y verificar **gradient accumulation** con equivalencia numérica.
7. Dominar **mixed precision** (fp32/fp16/bf16): bits, rangos, GradScaler, speedup.
8. Diagnosticar y remediar **gradient explosion** con clipping por norma global.
9. Entender el **vanishing** y cómo pre-LN + residuals lo mitigan.
10. Guardar y reanudar entrenamientos con **checkpoints completos**.
11. Implementar y comparar **LR schedulers**: warmup lineal + cosine decay.
12. Comparar **optimizadores**: AdamW, SGD+momentum, Adafactor, Lion.
13. Demo de **curriculum learning** (ordenar por dificultad).
14. Verificar **scaling laws** entrenando 3-4 tamaños en vivo.
15. Monitorear **hardware** (VRAM, throughput, MFU).
16. Seguir la **evolución interna** (pesos, embeddings PCA, accuracy).

## Prerrequisitos

- Notebooks 01–05 (tokenización, embeddings, arquitectura transformer)
- PyTorch ≥ 2.0, matplotlib, seaborn, plotly

## Tiempo estimado

| Perfil | Hardware | Tiempo |
|--------|----------|--------|
| SMALL  | T4 / CPU / MPS | ~60 min |
| LARGE  | A100 / L4      | ~90 min |


## Tabla de Contenidos

1. [Setup y configuración](#1-setup)
2. [Next-token prediction y teacher forcing](#2-ntp)
3. [Cross-entropy como MLE y perplexity](#3-ce)
4. [Datos: tokenización, packing, DataLoader](#4-datos)
5. [Training loop desde cero](#5-loop)
6. [Batch size y ruido del gradiente](#6-batch)
7. [Gradient accumulation](#7-accum)
8. [Mixed precision: fp32/fp16/bf16](#8-amp)
9. [Gradient clipping y explosión](#9-clip)
10. [Vanishing gradients y pre-LN](#10-vanish)
11. [Checkpointing completo y reanudación](#11-ckpt)
12. [LR schedulers: warmup + cosine decay](#12-lr)
13. [Optimizadores: AdamW, SGD, Adafactor, Lion](#13-optim)
14. [Curriculum learning](#14-curriculum)
15. [Scaling laws en vivo](#15-scaling)
16. [Monitoreo de hardware y MFU](#16-hw)
17. [Evolución interna: pesos, embeddings, accuracy](#17-evol)
18. [Resumen, ejercicios y referencias](#18-resumen)


In [ ]:
# ── Instalación (solo lo que no viene con Colab/torch) ──────────────────────
import subprocess, sys

def pip_install_quiet(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)

for pkg in ["plotly", "seaborn"]:
    try:
        __import__(pkg)
    except ImportError:
        pip_install_quiet(pkg)

# pynvml para monitoreo GPU (opcional)
try:
    import pynvml
    pynvml.nvmlInit()
    HAS_PYNVML = True
except Exception:
    HAS_PYNVML = False

print("Dependencias OK")


In [ ]:
# ── Detección de hardware ────────────────────────────────────────────────────
import os, platform, math, time, json, random, copy, itertools, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="darkgrid", palette="muted")
matplotlib.rcParams["figure.dpi"] = 110

# ── Seed global ──────────────────────────────────────────────────────────────
SEED = 42
def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed()

# ── Dispositivo ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16_ok  = torch.cuda.is_bf16_supported()
    print(f"GPU : {gpu_name}  |  VRAM: {vram_gb:.1f} GB  |  bf16: {bf16_ok}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE  = torch.device("mps")
    vram_gb = 0.0
    bf16_ok = False
    print("Dispositivo: Apple MPS")
else:
    DEVICE  = torch.device("cpu")
    vram_gb = 0.0
    bf16_ok = False
    print("Dispositivo: CPU (entrenamiento lento, SMALL obligatorio)")

# ── Colab ────────────────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
    print("Entorno: Google Colab")
except ImportError:
    IN_COLAB = False
    print("Entorno: local")

print(f"PyTorch {torch.__version__}  |  SEED={SEED}")


In [ ]:
# ── Configuración SMALL / LARGE ──────────────────────────────────────────────
# Forzar manualmente: PROFILE = "small" o PROFILE = "large"
PROFILE = "auto"

CONFIGS = {
    "small": dict(
        # Modelo ~0.5 M params con vocab_size=128 (char-level)
        vocab_size     = 128,
        d_model        = 128,
        n_layers       = 4,
        n_heads        = 4,
        d_ff           = 512,
        max_seq_len    = 128,
        dropout        = 0.1,
        # Entrenamiento
        batch_size     = 32,
        grad_accum     = 1,
        max_steps      = 1000,
        eval_every     = 200,
        lr             = 3e-4,
        weight_decay   = 0.1,
        warmup_steps   = 100,
        grad_clip      = 1.0,
        amp_dtype      = "fp16",   # T4 no soporta bf16
    ),
    "large": dict(
        vocab_size     = 128,
        d_model        = 256,
        n_layers       = 6,
        n_heads        = 8,
        d_ff           = 1024,
        max_seq_len    = 256,
        dropout        = 0.1,
        batch_size     = 64,
        grad_accum     = 4,
        max_steps      = 3000,
        eval_every     = 500,
        lr             = 6e-4,
        weight_decay   = 0.1,
        warmup_steps   = 300,
        grad_clip      = 1.0,
        amp_dtype      = "bf16",   # A100/L4
    ),
}

if PROFILE == "auto":
    if DEVICE.type == "cuda" and vram_gb >= 20:
        PROFILE = "large"
    else:
        PROFILE = "small"

CFG = CONFIGS[PROFILE]
print(f"Perfil seleccionado: {PROFILE.upper()}")
print(json.dumps(CFG, indent=2))


## MiniGPT — Modelo autocontenido

Definimos un **decoder-only Transformer compacto** (~100 líneas) que usaremos
a lo largo de todo el notebook. Incluye:

- **Pre-LayerNorm** (norma *antes* de la atención/FFN, no después — estilo GPT-2, Llama)
- **Multi-Head Causal Self-Attention** con máscara triangular
- **FFN** con activación GELU
- **Tied embeddings** (la cabeza de salida comparte pesos con la capa de embedding)
- **RoPE simplificado** (opcional, activable con `use_rope=True`)

La arquitectura pre-LN es crucial para el flujo de gradientes (§10).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MiniGPT — decoder-only Transformer (~100 líneas), completamente autocontenido
# ══════════════════════════════════════════════════════════════════════════════

class RMSNorm(nn.Module):
    """Root Mean Square Layer Norm (más eficiente que LayerNorm)."""
    def __init__(self, d: int, eps: float = 1e-6):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):  # x: (B, T, d)
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.g * x / rms


def make_causal_mask(T: int, device):
    """Máscara booleana causal (True = ignorar) de forma (T, T)."""
    return torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads
        self.qkv     = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.drop    = nn.Dropout(dropout)

    def forward(self, x):                           # x: (B, T, d_model)
        B, T, d = x.shape
        qkv = self.qkv(x)                           # (B, T, 3*d)
        q, k, v = qkv.split(d, dim=-1)             # cada (B, T, d)
        # reshape a (B, n_heads, T, d_head)
        def split_heads(t):
            return t.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = split_heads(q), split_heads(k), split_heads(v)
        # Scaled dot-product attention con máscara causal
        scale  = self.d_head ** -0.5
        scores = (q @ k.transpose(-2, -1)) * scale  # (B, h, T, T)
        mask   = make_causal_mask(T, x.device)       # (T, T)
        scores = scores.masked_fill(mask, float("-inf"))
        attn   = self.drop(scores.softmax(dim=-1))  # (B, h, T, T)
        out    = attn @ v                            # (B, h, T, d_head)
        out    = out.transpose(1, 2).contiguous().view(B, T, d)
        return self.out_proj(out)                    # (B, T, d_model)


class FFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.0):
        super().__init__()
        self.fc1  = nn.Linear(d_model, d_ff, bias=False)
        self.fc2  = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):  # (B, T, d_model) -> (B, T, d_model)
        return self.drop(self.fc2(F.gelu(self.fc1(x))))


class TransformerBlock(nn.Module):
    """Bloque con Pre-LayerNorm: norm → attn → residual, norm → ffn → residual."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ffn   = FFN(d_model, d_ff, dropout)

    def forward(self, x):          # (B, T, d_model)
        x = x + self.attn(self.norm1(x))   # residual + atención
        x = x + self.ffn(self.norm2(x))    # residual + FFN
        return x


class MiniGPT(nn.Module):
    """Decoder-only Transformer compacto (GPT-style, pre-LN).

    Args:
        vocab_size: tamaño del vocabulario
        d_model:    dimensión de los embeddings
        n_layers:   número de bloques Transformer
        n_heads:    cabezas de atención
        d_ff:       dimensión interna de la FFN
        max_seq_len: longitud máxima de secuencia
        dropout:    tasa de dropout
    """
    def __init__(self, vocab_size, d_model, n_layers, n_heads,
                 d_ff, max_seq_len=512, dropout=0.1):
        super().__init__()
        self.d_model    = d_model
        self.max_seq_len= max_seq_len
        self.tok_emb    = nn.Embedding(vocab_size, d_model)
        self.pos_emb    = nn.Embedding(max_seq_len, d_model)
        self.drop       = nn.Dropout(dropout)
        self.blocks     = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        self.norm_final = RMSNorm(d_model)
        # LM head — tied weights con tok_emb
        self.lm_head    = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

        # Inicialización: GPT-2 style (std ∝ 1/√(2*n_layers))
        self._init_weights()

    def _init_weights(self):
        std = 0.02 / math.sqrt(2 * max(len(self.blocks), 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, std=std)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)

    def forward(self, idx):        # idx: (B, T)  →  logits: (B, T, vocab_size)
        B, T = idx.shape
        assert T <= self.max_seq_len, f"seq_len {T} > max_seq_len {self.max_seq_len}"
        pos    = torch.arange(T, device=idx.device).unsqueeze(0)  # (1, T)
        x      = self.drop(self.tok_emb(idx) + self.pos_emb(pos)) # (B, T, d_model)
        for block in self.blocks:
            x  = block(x)                                          # (B, T, d_model)
        x      = self.norm_final(x)                                # (B, T, d_model)
        return self.lm_head(x)                                     # (B, T, vocab_size)

    def num_params(self, include_embeddings=True):
        if include_embeddings:
            return sum(p.numel() for p in self.parameters())
        # excluir embeddings de posición y token
        skip = {id(self.tok_emb.weight), id(self.pos_emb.weight)}
        return sum(p.numel() for p in self.parameters() if id(p) not in skip)


# ── Instanciar modelo con el perfil activo ───────────────────────────────────
model = MiniGPT(
    vocab_size  = CFG["vocab_size"],
    d_model     = CFG["d_model"],
    n_layers    = CFG["n_layers"],
    n_heads     = CFG["n_heads"],
    d_ff        = CFG["d_ff"],
    max_seq_len = CFG["max_seq_len"],
    dropout     = CFG["dropout"],
).to(DEVICE)

n_params = model.num_params()
print(f"MiniGPT ({PROFILE})  |  parámetros: {n_params:,}  (~{n_params/1e6:.2f} M)")
print(f"d_model={CFG['d_model']}, n_layers={CFG['n_layers']}, "
      f"n_heads={CFG['n_heads']}, d_ff={CFG['d_ff']}")


<a id="2-ntp"></a>
## 2. Next-token Prediction y Teacher Forcing

### 2.1 Formulación del problema

Un modelo de lenguaje autoregresivo factoriza la probabilidad de una secuencia
$\mathbf{x} = (x_1, x_2, \dots, x_T)$ como:

$$P(\mathbf{x}) = \prod_{t=1}^{T} P(x_t \mid x_1, \dots, x_{t-1})$$

El modelo aprende a predecir el siguiente token dado el contexto anterior.

### 2.2 Shift de targets (inputs vs. targets)

Dado el tensor de tokens `[x_0, x_1, ..., x_{T-1}]`:

```
inputs  =  x_0  x_1  x_2  ...  x_{T-2}   (los primeros T-1 tokens)
targets =  x_1  x_2  x_3  ...  x_{T-1}   (los siguientes T-1 tokens)
```

El modelo recibe `inputs` y produce logits para cada posición.
En la posición $t$, predice $P(x_{t+1} \mid x_0, \dots, x_t)$.

```
┌─────────────────────────────────────────────────────────┐
│  secuencia:   H  o  l  a     m  u  n  d  o              │
│  inputs:      H  o  l  a     m  u  n  d    → logits[0..8]│
│  targets:     o  l  a     m  u  n  d  o    → cross-ent. │
└─────────────────────────────────────────────────────────┘
```

### 2.3 Teacher Forcing

**Teacher forcing** = en cada paso $t$, el modelo recibe el token
*real* $x_t$ como contexto (no su propia predicción previa).

**Ventaja:** gradientes estables; permite **T predicciones en paralelo**
durante el entrenamiento (el forward pass es una sola operación matricial
sobre la secuencia completa, aprovechando la máscara causal).

**Desventaja — Exposure Bias:** en inferencia, el modelo consume sus propias
predicciones (que pueden ser incorrectas), creando una distribución que nunca
vio durante entrenamiento. Mitigation: *scheduled sampling*, *RLHF*, *DPO*.

### 2.4 ¿Por qué T predicciones en paralelo?

La máscara causal garantiza que el logit en posición $t$ solo depende de
tokens $0..t$ (y nunca del futuro). Por eso:

$$\text{logits}[t] = f_\theta(x_0, \dots, x_t)$$

con una sola pasada hacia adelante obtenemos los $T$ logits simultáneamente —
eficiencia clave que hace que el entrenamiento de LLMs sea factible.


In [ ]:
# ── Demostración del shift de targets ────────────────────────────────────────
def make_targets(ids: torch.Tensor) -> tuple:
    """Convierte una secuencia de IDs en (inputs, targets) con shift de 1."""
    return ids[..., :-1], ids[..., 1:]

# Ejemplo con una secuencia corta
seq = torch.arange(10)               # [0, 1, 2, ..., 9]
inp, tgt = make_targets(seq)
print("secuencia:", seq.tolist())
print("inputs   :", inp.tolist())
print("targets  :", tgt.tolist())
print("shift confirmado (inp+1 == tgt):", torch.all(inp + 1 == tgt).item())

# ── Diagrama ASCII del teacher forcing ───────────────────────────────────────
print("""
Teacher Forcing — forward pass con máscara causal:

  tokens:  t0   t1   t2   t3   t4
           ↓    ↓    ↓    ↓    ↓
  model: [GPT con máscara causal triangular]
           ↓    ↓    ↓    ↓    ↓
  logits: l0   l1   l2   l3   l4
               ↕    ↕    ↕    ↕
  targets:     t1   t2   t3   t4   (shift +1)
               ↕    ↕    ↕    ↕
  loss: CE(l0,t1) + CE(l1,t2) + ... + CE(l3,t4)  / (T-1)
""")


In [ ]:
# ── Visualización de la máscara causal ───────────────────────────────────────
T_demo = 8
mask = torch.triu(torch.ones(T_demo, T_demo), diagonal=1).numpy()

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(1 - mask, annot=True, fmt=".0f", cmap="Blues",
            vmin=0, vmax=1, ax=ax, cbar=False,
            linewidths=0.5, linecolor="gray")
ax.set_title("Máscara causal (1=atención permitida)", fontsize=12)
ax.set_xlabel("posición de clave (k)")
ax.set_ylabel("posición de query (q)")
for i in range(T_demo):
    ax.text(i + 0.5, i + 0.5, "●", ha="center", va="center",
            fontsize=10, color="crimson")
plt.tight_layout()
plt.savefig("/tmp/causal_mask.png", dpi=100, bbox_inches="tight")
plt.show()
print("● = diagonal (auto-atención)")


<a id="3-ce"></a>
## 3. Cross-Entropy como MLE y Perplexity

### 3.1 Derivación desde Máxima Verosimilitud

Dado el corpus $\mathcal{D} = \{x^{(i)}\}$, maximizar la log-verosimilitud:

$$\mathcal{L}_{\text{MLE}} = \frac{1}{N} \sum_{i} \sum_{t} \log P_\theta(x_t^{(i)} \mid x_{<t}^{(i)})$$

equivale a **minimizar la cross-entropy promedio** entre la distribución
del modelo $q_\theta$ y la distribución empírica $p$ (one-hot):

$$H(p, q_\theta) = -\sum_v p(v) \log q_\theta(v) = -\log q_\theta(x_t^*)$$

donde $x_t^*$ es el token correcto.

En PyTorch: `nn.CrossEntropyLoss` aplica `log_softmax` + `nll_loss`,
trabajando en logits (numéricamente estable).

### 3.2 `ignore_index`

En secuencias con padding, los tokens de relleno (`PAD`) no deben contribuir
a la pérdida. `ignore_index=-100` indica a PyTorch que ignore esas posiciones:

```python
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
```

Con **packing** (§4) eliminamos el padding: todas las posiciones son tokens reales.

### 3.3 Label Smoothing

Reemplaza los targets one-hot por una distribución suavizada:
$y_i = (1-\epsilon)\, \delta_{i,\text{gold}} + \epsilon / V$

- Reduce sobre-confianza; útil en **fine-tuning** donde el vocabulario correcto
  está bien definido.
- En **pre-training** apenas ayuda: el corpus es tan ruidoso que la incertidumbre
  ya está presente; además aumenta el costo computacional.

### 3.4 Perplexity

La *perplexity* (PPL) mide cuántos tokens "sospecha" el modelo en cada posición:

$$\text{PPL} = \exp\!\left(\frac{1}{T}\sum_t -\log P_\theta(x_t \mid x_{<t})\right) = \exp(\mathcal{L})$$

- PPL = 1 → modelo perfecto (predice con certeza el token correcto).
- PPL = $V$ → peor que el azar uniforme.
- En práctica: GPT-2 logra ~29 en WikiText-103; modelos modernos < 10.


In [ ]:
# ── Cross-entropy vs. log_softmax + nll_loss (verificación numérica) ─────────
torch.manual_seed(0)
B, T, V = 2, 5, CFG["vocab_size"]
logits  = torch.randn(B, T, V)   # (B, T, V)
targets = torch.randint(0, V, (B, T))  # (B, T)

# Forma estándar de PyTorch (reshape para CrossEntropyLoss)
loss_torch = F.cross_entropy(logits.view(-1, V), targets.view(-1))

# Derivación manual: log_softmax + nll
log_probs   = F.log_softmax(logits, dim=-1)          # (B, T, V)
nll         = -log_probs[torch.arange(B)[:, None],   # indexing vectorizado
                          torch.arange(T)[None, :],
                          targets]                    # (B, T)
loss_manual = nll.mean()

print(f"CrossEntropyLoss (torch)  : {loss_torch.item():.6f}")
print(f"log_softmax + NLL (manual): {loss_manual.item():.6f}")
assert torch.allclose(loss_torch, loss_manual, atol=1e-5), "¡Discrepancia!"
print("✓ Equivalencia verificada")

# ── Perplexity ────────────────────────────────────────────────────────────────
ppl = torch.exp(loss_torch)
print(f"\nPerplexity  = exp({loss_torch.item():.4f}) = {ppl.item():.2f}")
print(f"(vocabulario={V}; PPL uniforme = {V})")

# ── Label smoothing ───────────────────────────────────────────────────────────
ls_values = [0.0, 0.05, 0.1, 0.2]
losses_ls = []
for eps in ls_values:
    loss_ls = F.cross_entropy(logits.view(-1, V), targets.view(-1), label_smoothing=eps)
    losses_ls.append(loss_ls.item())

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(ls_values, losses_ls, "o-", color="steelblue")
ax.set_xlabel("label_smoothing ε")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("Efecto del label smoothing (random logits)")
plt.tight_layout(); plt.show()
print("Nota: con logits aleatorios, el smoothing aumenta la loss porque")
print("distribuye probabilidad a tokens incorrectos.")


<a id="4-datos"></a>
## 4. Datos: Tokenización, Packing y DataLoader

### 4.1 Tokenización a nivel de carácter

Para este laboratorio usamos un **tokenizador de caracteres** (char-level):
cada carácter ASCII mapea a un ID entero. Vocabulario = 128 tokens.
Simple, sin dependencias, y perfectamente suficiente para demostrar todos
los conceptos del pipeline de entrenamiento.

En producción se usa BPE byte-level (Notebook 02), pero la mecánica del
entrenamiento es idéntica.

### 4.2 Padding vs. Packing

**Padding:** cada secuencia tiene longitud fija; las más cortas se rellenan
con tokens `[PAD]` que se ignoran con `ignore_index`. Desperdicia cómputo.

**Packing:** se concatenan documentos (separados por `[EOS]`) en chunks
contiguos de longitud `block_size`. **No hay tokens desperdiciados.**
Es la estrategia estándar en pre-training (GPT, Llama, etc.).

```
[doc1_tokens] [EOS] [doc2_tokens] [EOS] [doc3_tok... | chunk límite]
└─────────── chunk 0 ───────────────────────────────┘
                                         └─────────── chunk 1 ──────...
```

### 4.3 Dataset con packing

```python
dataset[i]  →  tensor de shape (block_size,)
targets[i]  →  tensor de shape (block_size,)  # shift +1
```


In [ ]:
# ── Corpus técnico inline (texto en español/inglés sobre LLMs) ───────────────
CORPUS_DOCS = [
    "Los modelos de lenguaje aprenden a predecir el siguiente token dado el contexto anterior. "
    "Este proceso se denomina modelado autoregresivo y es la base de GPT, Llama y Claude.",

    "The transformer architecture uses self-attention to model long-range dependencies. "
    "Each token attends to all previous tokens through a causal mask.",

    "El entrenamiento con gradiente descendente estocástico actualiza los parámetros del modelo "
    "en la dirección que reduce la función de pérdida. AdamW combina momentum y regularización.",

    "Backpropagation computes gradients by applying the chain rule through the computational graph. "
    "Vanishing gradients occur when these products become exponentially small.",

    "La tasa de aprendizaje determina el tamaño del paso en cada actualización. "
    "Un warmup inicial estabiliza el entrenamiento en las primeras iteraciones.",

    "Mixed precision training uses fp16 or bf16 for forward and backward passes, "
    "keeping fp32 master weights for numerical stability in the optimizer.",

    "Gradient clipping previene que gradientes explosivos desestabilicen el entrenamiento. "
    "Se clipea la norma global del gradiente a un valor máximo.",

    "El batch size afecta el ruido del gradiente: batches grandes reducen la varianza "
    "pero requieren más memoria y pueden converger a mínimos agudos.",

    "Checkpointing allows resuming training from an exact state. "
    "A complete checkpoint saves model, optimizer, scheduler, scaler and RNG states.",

    "Las leyes de escala de Kaplan et al. muestran que la pérdida decrece como ley de potencias "
    "con el número de parámetros, tokens y cómputo de entrenamiento.",

    "Curriculum learning organizes training examples from easy to hard, "
    "potentially improving convergence speed and final performance.",

    "The cosine learning rate schedule decays the learning rate following a cosine curve "
    "after a linear warmup phase, which is standard in modern LLM training.",

    "AdamW separa el weight decay del término de momento, a diferencia de Adam original. "
    "Esto mejora la regularización en transformers con tied embeddings.",

    "Gradient accumulation simulates larger batch sizes by accumulating gradients "
    "over multiple micro-batches before performing a single optimizer step.",

    "Pre-layer normalization places the normalization before attention and FFN sublayers, "
    "improving gradient flow and training stability compared to post-LN.",
] * 8  # repetir para tener suficientes tokens

EOS_TOKEN = "\x03"   # carácter de control como separador de documentos

# ── Tokenizador char-level ────────────────────────────────────────────────────
class CharTokenizer:
    def __init__(self, vocab_size: int = 128):
        self.vocab_size = vocab_size  # ASCII 0-127

    def encode(self, text: str) -> list:
        return [ord(c) % self.vocab_size for c in text]

    def decode(self, ids: list) -> str:
        return "".join(chr(i) for i in ids if 32 <= i < 127)

tokenizer = CharTokenizer(vocab_size=CFG["vocab_size"])

# Tokenizar y concatenar documentos
all_tokens = []
for doc in CORPUS_DOCS:
    all_tokens.extend(tokenizer.encode(doc))
    all_tokens.append(tokenizer.encode(EOS_TOKEN)[0])

print(f"Total tokens en corpus: {len(all_tokens):,}")
print(f"Primeros 30 IDs: {all_tokens[:30]}")
print(f"Decodificado:    {tokenizer.decode(all_tokens[:30])!r}")


In [ ]:
# ── Dataset con packing ───────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

class PackedDataset(Dataset):
    """Divide el corpus en chunks contiguos de tamaño block_size (packing)."""

    def __init__(self, token_ids: list, block_size: int):
        data  = torch.tensor(token_ids, dtype=torch.long)
        # Recortar a múltiplo exacto de block_size
        n_chunks = len(data) // block_size
        data     = data[:n_chunks * block_size]
        self.chunks = data.view(n_chunks, block_size)  # (n_chunks, block_size)
        print(f"PackedDataset: {n_chunks} chunks × {block_size} tokens "
              f"= {n_chunks * block_size:,} tokens totales")

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]          # (block_size,)
        x     = chunk[:-1]               # inputs  (block_size-1,)
        y     = chunk[1:]                # targets (block_size-1,)
        return x, y

# split 90/10
block_size = CFG["max_seq_len"]
split_idx  = int(len(all_tokens) * 0.9)
train_tokens = all_tokens[:split_idx]
val_tokens   = all_tokens[split_idx:]

train_ds = PackedDataset(train_tokens, block_size)
val_ds   = PackedDataset(val_tokens,   block_size)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"],
                          shuffle=True, drop_last=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"],
                          shuffle=False, drop_last=False, num_workers=0)

print(f"\nTrain batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# ── Medir tokens/segundo del DataLoader ───────────────────────────────────────
t0 = time.time()
total_toks = 0
for x, y in itertools.islice(train_loader, 20):
    total_toks += x.numel()
elapsed = time.time() - t0
print(f"\nDataLoader throughput: {total_toks / elapsed:,.0f} tokens/seg "
      f"(sobre 20 batches, CPU)")


In [ ]:
# ── Visualización de packing vs padding ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))

# Padding (simulado)
block = 16
docs  = [8, 4, 12, 6, 10, 5, 9, 7]
pad_matrix = np.zeros((len(docs), block))
for i, dlen in enumerate(docs):
    pad_matrix[i, :dlen] = 1.0

axes[0].imshow(pad_matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[0].set_title("Padding (verde=token, rojo=PAD)")
axes[0].set_xlabel("posición en secuencia"); axes[0].set_ylabel("ejemplo")

# Packing
all_d = sum(docs)
pack_array = np.ones(all_d)
pos = 0
boundaries = []
for dlen in docs:
    boundaries.append(pos)
    pos += dlen
pack_matrix = pack_array[:block * (all_d // block)].reshape(-1, block)
axes[1].imshow(pack_matrix, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
axes[1].set_title("Packing (100% utilización)")
axes[1].set_xlabel("posición en chunk"); axes[1].set_ylabel("chunk")

plt.suptitle("Padding vs. Packing", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

utilization_pad  = np.mean(pad_matrix)
utilization_pack = 1.0
print(f"Utilización padding: {utilization_pad:.1%}  |  Packing: {utilization_pack:.1%}")


<a id="5-loop"></a>
## 5. Training Loop desde Cero

### 5.1 Estructura general

```
for step in range(max_steps):
    x, y = next(train_iter)          # batch
    loss = train_step(model, x, y)   # forward + backward + step
    if step % eval_every == 0:
        val_loss = eval_step(model, val_loader)
        metrics_logger.log(step, loss=loss, val_loss=val_loss, ...)
```

### 5.2 `model.eval()` y `model.train()`

- **`model.eval()`**: desactiva Dropout (todas las unidades activas con su valor
  escalado) y pone BatchNorm en modo de inferencia (usa estadísticas acumuladas).
  MiniGPT solo tiene Dropout, pero el principio es universal.
- **`torch.no_grad()`**: desactiva el grafo computacional durante evaluación →
  ahorra ~50% de memoria y acelera el cómputo.
- **`model.train()`**: reactiva Dropout y BN para el siguiente epoch.

### 5.3 MetricsLogger (patrón JSONL)

Cada punto de métricas se escribe como JSON Lines:
```json
{"step": 0, "wall_time": 0.0, "loss": 4.82, "val_loss": 4.91, "ppl": 123.5}
```
Reconstruir curvas: `pd.read_json("logs/metrics_run.jsonl", lines=True)`.


In [ ]:
# ── MetricsLogger (patrón del proyecto, inline) ───────────────────────────────
import json, time
from pathlib import Path

class MetricsLogger:
    """Acumula métricas por paso y las persiste como JSON Lines."""
    def __init__(self, run_name: str, log_dir: str = "logs"):
        self.path = Path(log_dir) / f"metrics_{run_name}.jsonl"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._t0  = time.time()
        self.history: list = []
        # limpiar archivo previo si existe
        self.path.write_text("")

    def log(self, step: int, **metrics):
        record = {"step": step,
                  "wall_time": round(time.time() - self._t0, 2),
                  **{k: round(float(v), 6) for k, v in metrics.items()}}
        self.history.append(record)
        with self.path.open("a") as f:
            f.write(json.dumps(record) + "\n")

    def to_dataframe(self):
        import pandas as pd
        return pd.DataFrame(self.history)

print("MetricsLogger definido")


In [ ]:
# ── Training loop completo ────────────────────────────────────────────────────
set_seed(SEED)

# Re-instanciar modelo limpio para este entrenamiento base
model_base = MiniGPT(
    vocab_size  = CFG["vocab_size"],
    d_model     = CFG["d_model"],
    n_layers    = CFG["n_layers"],
    n_heads     = CFG["n_heads"],
    d_ff        = CFG["d_ff"],
    max_seq_len = CFG["max_seq_len"],
    dropout     = CFG["dropout"],
).to(DEVICE)

optimizer_base = torch.optim.AdamW(
    model_base.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"],
    betas=(0.9, 0.95),
)

loss_fn = nn.CrossEntropyLoss()

# ── train_step ────────────────────────────────────────────────────────────────
def train_step(model, optimizer, x, y):
    """Un paso de entrenamiento: forward → loss → backward → optimizer step."""
    model.train()
    x, y   = x.to(DEVICE), y.to(DEVICE)
    logits = model(x)                                  # (B, T, V)
    loss   = loss_fn(logits.view(-1, CFG["vocab_size"]), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
    optimizer.step()
    return loss.item()

# ── eval_step ─────────────────────────────────────────────────────────────────
@torch.no_grad()
def eval_step(model, loader, max_batches=30):
    """Evalúa el modelo en un loader; devuelve (avg_loss, perplexity)."""
    model.eval()
    total_loss = 0.0
    n_batches  = 0
    for x, y in itertools.islice(loader, max_batches):
        x, y   = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        loss   = loss_fn(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        total_loss += loss.item()
        n_batches  += 1
    avg_loss = total_loss / max(n_batches, 1)
    return avg_loss, math.exp(avg_loss)

# ── Loop principal ────────────────────────────────────────────────────────────
MAX_STEPS_BASE = min(CFG["max_steps"], 1000)  # cap para no tardar demasiado
EVAL_EVERY     = CFG["eval_every"]

logger_base = MetricsLogger("base_run")
train_iter  = itertools.cycle(train_loader)

print(f"Entrenando {MAX_STEPS_BASE} pasos | PROFILE={PROFILE} | device={DEVICE}")
t_start = time.time()

for step in range(MAX_STEPS_BASE):
    x, y  = next(train_iter)
    loss  = train_step(model_base, optimizer_base, x, y)

    if step % EVAL_EVERY == 0 or step == MAX_STEPS_BASE - 1:
        val_loss, val_ppl = eval_step(model_base, val_loader)
        tokens_seen = (step + 1) * CFG["batch_size"] * (CFG["max_seq_len"] - 1)
        elapsed     = time.time() - t_start
        toks_per_s  = tokens_seen / elapsed if elapsed > 0 else 0
        logger_base.log(step, loss=loss, val_loss=val_loss, val_ppl=val_ppl,
                        tokens_seen=tokens_seen, toks_per_s=toks_per_s)
        print(f"step {step:5d} | loss {loss:.4f} | val_loss {val_loss:.4f} "
              f"| val_ppl {val_ppl:.1f} | {toks_per_s:,.0f} tok/s")

print(f"\nEntrenamiento terminado en {time.time()-t_start:.1f}s")


In [ ]:
# ── Curvas de entrenamiento ───────────────────────────────────────────────────
df_base = logger_base.to_dataframe()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(df_base["step"], df_base["loss"], label="train loss", alpha=0.7)
axes[0].plot(df_base["step"], df_base["val_loss"], label="val loss",
             linewidth=2, color="crimson")
axes[0].set_xlabel("paso"); axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Curvas de pérdida"); axes[0].legend()

axes[1].plot(df_base["step"], df_base["val_ppl"], color="darkorange", linewidth=2)
axes[1].set_xlabel("paso"); axes[1].set_ylabel("Perplexity")
axes[1].set_title("Perplexity de validación")
axes[1].axhline(CFG["vocab_size"], ls="--", color="gray", label=f"PPL uniforme={CFG['vocab_size']}")
axes[1].legend()

plt.suptitle(f"Training Run Base — {PROFILE.upper()}", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


<a id="6-batch"></a>
## 6. Batch Size y Ruido del Gradiente

### 6.1 Varianza del gradiente estocástico

El gradiente calculado sobre un mini-batch de tamaño $B$ es un estimador
insesgado del gradiente verdadero, con varianza:

$$\text{Var}[\hat{g}_B] = \frac{\sigma^2}{B}$$

donde $\sigma^2$ es la varianza del gradiente de un solo ejemplo.

- **Batch pequeño** ($B=1$): mucho ruido → exploración amplia del espacio de parámetros,
  puede escapar de mínimos locales, pero los pasos son ineficientes.
- **Batch grande** ($B \gg 1$): gradientes suaves → convergencia estable pero menos
  exploración; requiere LR más alta para compensar.

### 6.2 Critical Batch Size

Existe un *critical batch size* $B_{\text{crit}}$ donde el ruido de gradiente
es comparable a la señal. Incrementar $B$ más allá de $B_{\text{crit}}$ da
rendimientos decrecientes en eficiencia muestral:

$$B_{\text{crit}} \approx \frac{\sigma^2}{\|\nabla \mathcal{L}\|^2}$$

(McCandlish et al., 2018). En la práctica: para GPT-3 es $\sim 10^6$ tokens.

### 6.3 Compromiso memoria/velocidad

- **Memoria**: $\propto B \cdot T \cdot d_{\text{model}}$ (activaciones).
- **Velocidad**: el hardware tiene un "batch size óptimo" para maximizar utilización.
- **Gradiente acumulación** (§7): simula batches grandes sin usar más VRAM.


In [ ]:
# ── Experimento: varianza del gradiente vs. batch size ────────────────────────
set_seed(SEED)

# Modelo pequeño fijo para el experimento
model_exp = MiniGPT(
    vocab_size=CFG["vocab_size"], d_model=64, n_layers=2,
    n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
).to(DEVICE)

batch_sizes = [1, 4, 8, 16, 32, 64]
N_TRIALS    = 20   # gradientes independientes por B

grad_variances = {}

for B in batch_sizes:
    norms = []
    # Crear loader con batch size B
    bs_loader = DataLoader(train_ds, batch_size=B, shuffle=True,
                           drop_last=True, num_workers=0)
    bs_iter   = iter(bs_loader)
    for _ in range(N_TRIALS):
        try:
            x, y = next(bs_iter)
        except StopIteration:
            bs_iter = iter(bs_loader)
            x, y   = next(bs_iter)
        x, y   = x.to(DEVICE), y.to(DEVICE)
        logits = model_exp(x)
        loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        model_exp.zero_grad()
        loss.backward()
        # norma global del gradiente
        total_norm = 0.0
        for p in model_exp.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        norms.append(total_norm ** 0.5)
    grad_variances[B] = (np.mean(norms), np.std(norms))
    print(f"B={B:3d}: media_norma={np.mean(norms):.4f}  std={np.std(norms):.4f}")

# ── Gráfica ───────────────────────────────────────────────────────────────────
Bs   = list(grad_variances.keys())
meds = [grad_variances[b][0] for b in Bs]
stds = [grad_variances[b][1] for b in Bs]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].errorbar(Bs, meds, yerr=stds, fmt="o-", capsize=5, color="steelblue")
axes[0].set_xlabel("Batch size B"); axes[0].set_ylabel("Norma del gradiente")
axes[0].set_title("Norma media ± std del gradiente vs. B")
axes[0].set_xscale("log", base=2)

# Varianza relativa (std/mean) ∝ 1/sqrt(B)
rel_var = [s/m for m, s in zip(meds, stds)]
theory  = [rel_var[0] / math.sqrt(b / Bs[0]) for b in Bs]
axes[1].plot(Bs, rel_var, "o-", label="empírico", color="crimson")
axes[1].plot(Bs, theory, "--", label=r"$\propto 1/\sqrt{B}$", color="gray")
axes[1].set_xlabel("Batch size B"); axes[1].set_ylabel("std / media")
axes[1].set_title("Ruido relativo del gradiente"); axes[1].legend()
axes[1].set_xscale("log", base=2)

plt.suptitle("Ruido del gradiente vs. Batch Size", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


<a id="7-accum"></a>
## 7. Gradient Accumulation

### 7.1 Equivalencia matemática

Gradient accumulation simula un batch efectivo $B_{\text{eff}} = B \times K$
(donde $K$ es el número de micro-batches) sin aumentar el uso de VRAM.

**Derivación:** el gradiente sobre el batch completo es:

$$\nabla_\theta \mathcal{L}_{B_{\text{eff}}} = \frac{1}{B_{\text{eff}}} \sum_{i=1}^{B_{\text{eff}}} \nabla_\theta \ell_i = \frac{1}{K} \sum_{k=1}^{K} \underbrace{\frac{1}{B} \sum_{i \in \text{micro}_k} \nabla_\theta \ell_i}_{= \nabla_\theta \mathcal{L}_{\text{micro}_k}}$$

Por lo tanto: **acumular gradientes de $K$ micro-batches y dividir por $K$**
es exactamente equivalente a calcularlos sobre el batch completo.

En código:
```python
for k, (x, y) in enumerate(micro_batches):
    loss = criterion(model(x), y) / K   # ← dividir antes de backward
    loss.backward()                      # gradientes se ACUMULAN
optimizer.step()
optimizer.zero_grad()
```

**Cuidado con la normalización**: PyTorch promedia la loss por defecto
(`reduction="mean"`). Al dividir por $K$ antes del `backward`, el gradiente
acumulado equivale exactamente al del batch completo.


In [ ]:
# ── Implementación de gradient accumulation ──────────────────────────────────
set_seed(SEED)

# Modelo y datos fijos para comparación exacta
torch.manual_seed(SEED)
model_accum = MiniGPT(
    vocab_size=CFG["vocab_size"], d_model=64, n_layers=2,
    n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
).to(DEVICE)

# Guardar estado inicial
state_init = copy.deepcopy(model_accum.state_dict())

# ── Método A: batch completo B_eff ────────────────────────────────────────────
K       = 4  # acumulación steps
B_micro = 8  # micro-batch
B_eff   = B_micro * K

# Construir un batch grande fijo
loader_big = DataLoader(train_ds, batch_size=B_eff, shuffle=False,
                        drop_last=True, num_workers=0)
x_big, y_big = next(iter(loader_big))
x_big, y_big = x_big.to(DEVICE), y_big.to(DEVICE)

opt_A = torch.optim.SGD(model_accum.parameters(), lr=0.01)
model_accum.load_state_dict(state_init)
opt_A.zero_grad()
logits_big = model_accum(x_big)
loss_A     = F.cross_entropy(logits_big.view(-1, CFG["vocab_size"]), y_big.view(-1))
loss_A.backward()
# gradientes del método A
grads_A = [p.grad.clone() for p in model_accum.parameters() if p.grad is not None]

# ── Método B: gradient accumulation sobre K micro-batches ─────────────────────
model_accum.load_state_dict(state_init)
opt_B = torch.optim.SGD(model_accum.parameters(), lr=0.01)
opt_B.zero_grad()

for k in range(K):
    x_k = x_big[k*B_micro:(k+1)*B_micro]
    y_k = y_big[k*B_micro:(k+1)*B_micro]
    logits_k = model_accum(x_k)
    loss_k   = F.cross_entropy(logits_k.view(-1, CFG["vocab_size"]),
                                y_k.view(-1)) / K   # ← dividir por K
    loss_k.backward()                               # acumular

grads_B = [p.grad.clone() for p in model_accum.parameters() if p.grad is not None]

# ── Verificación numérica ─────────────────────────────────────────────────────
max_diff = max(
    (gA - gB).abs().max().item()
    for gA, gB in zip(grads_A, grads_B)
)
print(f"Diferencia máxima entre gradientes (A vs B): {max_diff:.2e}")
equiv = all(torch.allclose(gA, gB, atol=1e-5) for gA, gB in zip(grads_A, grads_B))
print(f"Equivalencia numérica (allclose, atol=1e-5): {equiv}")
print(f"✓ Gradient accumulation verificado" if equiv else "✗ Discrepancia detectada")
print(f"\nBatch efectivo: {B_eff} = {K} micro-batches × {B_micro}")
print(f"VRAM máxima simultánea: proporcional a B_micro={B_micro} (no a B_eff={B_eff})")


<a id="8-amp"></a>
## 8. Mixed Precision: fp32 / fp16 / bf16

### 8.1 Representación de punto flotante

| Formato | Bits totales | Signo | Exponente | Mantisa | Rango approx. | Precisión |
|---------|-------------|-------|-----------|---------|----------------|-----------|
| fp32    | 32          | 1     | 8         | 23      | ±3.4×10³⁸     | ~7 dígitos |
| fp16    | 16          | 1     | 5         | 10      | ±65 504        | ~3 dígitos |
| bf16    | 16          | 1     | 8         | 7       | ±3.4×10³⁸     | ~2 dígitos |

**fp16**: rango dinámico pequeño → los gradientes pequeños hacen *underflow* a cero.
**bf16**: mismo exponente que fp32 → no hay underflow de gradientes; preferido en A100/H100.

### 8.2 ¿Por qué fp16 necesita GradScaler?

El rango de fp16 es ±65 504. Los gradientes típicos son $\sim 10^{-3}$ a $10^{-6}$,
que caben en fp16, **pero** gradients muy pequeños pueden hacer *underflow* (→ 0).

**Loss Scaling (dinámico)**: multiplicar la loss por un factor grande $S$ antes del
`backward` → los gradientes se amplían → caben en fp16 → dividir por $S$ antes del
optimizer step. `GradScaler` ajusta $S$ automáticamente.

**bf16 no necesita GradScaler** porque su rango es igual al de fp32.

### 8.3 Estrategia AMP (Automatic Mixed Precision)

```python
with torch.autocast(device_type="cuda", dtype=torch.float16):
    logits = model(x)          # forward en fp16
    loss   = criterion(...)    # loss en fp32 (sum acumulada en fp32)

scaler.scale(loss).backward()  # backward en fp16 (escalada)
scaler.step(optimizer)         # unscale + check NaN → optimizer step
scaler.update()                # ajustar S para el próximo paso
```


In [ ]:
# ── Tabla de formatos de punto flotante ──────────────────────────────────────
import struct

def fp_info(fmt):
    """Rango y eps para distintos formatos."""
    if fmt == "fp32":
        info = torch.finfo(torch.float32)
    elif fmt == "fp16":
        info = torch.finfo(torch.float16)
    elif fmt == "bf16":
        info = torch.finfo(torch.bfloat16)
    return info.max, info.tiny, info.eps

formats = ["fp32", "fp16", "bf16"]
print(f"{'Formato':<8} {'Max':<15} {'Min pos':<15} {'Epsilon':<15}")
print("-" * 55)
for fmt in formats:
    mx, mn, eps = fp_info(fmt)
    print(f"{fmt:<8} {mx:<15.3e} {mn:<15.3e} {eps:<15.3e}")

# ── Demostración de underflow en fp16 ────────────────────────────────────────
print("\n--- Underflow en fp16 ---")
val_fp32 = torch.tensor(1e-8, dtype=torch.float32)
val_fp16 = val_fp32.to(torch.float16)
val_bf16 = val_fp32.to(torch.bfloat16)
print(f"fp32: {val_fp32.item():.2e}  →  fp16: {val_fp16.item():.2e}  "
      f"|  bf16: {val_bf16.item():.2e}")
print(f"fp16 underflow: {val_fp16.item() == 0.0}")
print(f"bf16 underflow: {val_bf16.item() == 0.0}")


In [ ]:
# ── AMP: autocast + GradScaler ───────────────────────────────────────────────
set_seed(SEED)

# Solo medir si hay GPU (en CPU el speedup es negligible/negativo)
if DEVICE.type == "cuda":
    # Elegir dtype según soporte
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    print(f"AMP dtype: {amp_dtype}  |  GradScaler: {not bf16_ok}")

    model_amp = MiniGPT(
        vocab_size=CFG["vocab_size"], d_model=CFG["d_model"],
        n_layers=CFG["n_layers"], n_heads=CFG["n_heads"],
        d_ff=CFG["d_ff"], max_seq_len=CFG["max_seq_len"], dropout=0.0
    ).to(DEVICE)

    opt_amp   = torch.optim.AdamW(model_amp.parameters(), lr=CFG["lr"])
    scaler    = torch.cuda.amp.GradScaler(enabled=(amp_dtype == torch.float16))
    loader_amp= DataLoader(train_ds, batch_size=CFG["batch_size"],
                           shuffle=True, drop_last=True, num_workers=0)
    amp_iter  = itertools.cycle(loader_amp)

    def run_steps(model, optimizer, scaler, n_steps, use_amp):
        model.train()
        t0 = time.time()
        vram_start = torch.cuda.memory_allocated() / 1e6
        for _ in range(n_steps):
            x, y = next(amp_iter)
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            if use_amp:
                with torch.autocast(device_type="cuda", dtype=amp_dtype):
                    logits = model(x)
                    loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]),
                                             y.view(-1))
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(x)
                loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
                loss.backward()
                optimizer.step()
        torch.cuda.synchronize()
        elapsed = time.time() - t0
        vram_end = torch.cuda.memory_allocated() / 1e6
        return elapsed, vram_end - vram_start

    N_BENCH = 50
    t_fp32, v_fp32 = run_steps(model_amp, opt_amp, scaler, N_BENCH, use_amp=False)
    model_amp._init_weights()
    t_amp,  v_amp  = run_steps(model_amp, opt_amp, scaler, N_BENCH, use_amp=True)

    speedup = t_fp32 / t_amp
    print(f"\nfp32:  {t_fp32:.2f}s  |  AMP ({amp_dtype}): {t_amp:.2f}s")
    print(f"Speedup: {speedup:.2f}x")
    print(f"ΔVRAM fp32: {v_fp32:+.1f} MB  |  ΔVRAM AMP: {v_amp:+.1f} MB")

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].bar(["fp32", f"AMP ({amp_dtype})"], [t_fp32, t_amp],
                color=["steelblue", "darkorange"])
    axes[0].set_ylabel("Tiempo (s)"); axes[0].set_title(f"Tiempo ({N_BENCH} pasos)")
    axes[1].bar(["fp32", f"AMP ({amp_dtype})"], [abs(v_fp32), abs(v_amp)],
                color=["steelblue", "darkorange"])
    axes[1].set_ylabel("ΔVRAM (MB)"); axes[1].set_title("Memoria activaciones")
    plt.suptitle("Mixed Precision vs. fp32", fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    amp_dtype = torch.float32
    scaler    = torch.cuda.amp.GradScaler(enabled=False)
    print("Sin GPU → AMP no aplica. Se usa fp32.")
    print("En GPU: AMP con fp16/bf16 típicamente da 1.5-3x speedup y -30-50% VRAM.")


<a id="9-clip"></a>
## 9. Gradient Clipping y Explosión de Gradientes

### 9.1 Explosión de gradientes

Con **learning rate alta** o redes profundas sin normalización, los gradientes
se amplifican durante el backprop:

$$\|\nabla_\theta \mathcal{L}\| \gg 1 \quad \Rightarrow \quad \theta_{\text{new}} = \theta - \alpha \nabla \mathcal{L} \text{ puede diverger}$$

El síntoma en la loss: sube a `nan` o `inf` abruptamente.

### 9.2 Clipping por norma global

En lugar de clipear cada parámetro individualmente, se clipea la **norma global**:

$$\text{Si } \|\mathbf{g}\|_2 > \text{max\_norm}: \quad \mathbf{g} \leftarrow \mathbf{g} \cdot \frac{\text{max\_norm}}{\|\mathbf{g}\|_2}$$

Esto preserva la *dirección* del gradiente y solo reduce su magnitud.

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

### 9.3 Monitoreo como diagnóstico

Loguear la norma del gradiente **antes** del clipping es el diagnóstico más
informativo del estado del entrenamiento:
- Norma estable $\sim 0.1$–$1.0$: entrenamiento sano.
- Norma disparándose: LR demasiado alta, batch muy pequeño, o activaciones saturadas.
- Norma $\to 0$: vanishing (ver §10).


In [ ]:
# ── Experimento: provocar y curar explosión de gradientes ────────────────────
set_seed(SEED)

model_explode = MiniGPT(
    vocab_size=CFG["vocab_size"], d_model=64, n_layers=3,
    n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
).to(DEVICE)

loader_exp = DataLoader(train_ds, batch_size=4, shuffle=True,
                        drop_last=True, num_workers=0)
exp_iter   = itertools.cycle(loader_exp)

def run_with_without_clip(model, lr, n_steps, clip_val=None):
    """Entrena n_steps y registra norma del gradiente y loss."""
    model_c   = copy.deepcopy(model)
    opt_c     = torch.optim.SGD(model_c.parameters(), lr=lr)
    norms, losses = [], []
    for _ in range(n_steps):
        x, y = next(exp_iter)
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt_c.zero_grad()
        logits = model_c(x)
        loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        if not torch.isfinite(loss):
            norms.append(float("nan")); losses.append(float("nan"))
            continue
        loss.backward()
        # calcular norma ANTES del clip
        total_norm = 0.0
        for p in model_c.parameters():
            if p.grad is not None:
                total_norm += p.grad.norm(2).item() ** 2
        norms.append(total_norm ** 0.5)
        if clip_val is not None:
            torch.nn.utils.clip_grad_norm_(model_c.parameters(), clip_val)
        opt_c.step()
        losses.append(loss.item())
    return norms, losses

N_EXP = 80
HIGH_LR = 5.0  # LR extremadamente alta para provocar explosión

print(f"Entrenando con LR={HIGH_LR} (sin clip)...")
norms_nc, losses_nc = run_with_without_clip(model_explode, HIGH_LR, N_EXP, clip_val=None)

print(f"Entrenando con LR={HIGH_LR} (clip=1.0)...")
norms_cl, losses_cl = run_with_without_clip(model_explode, HIGH_LR, N_EXP, clip_val=1.0)

# ── Gráfica ───────────────────────────────────────────────────────────────────
steps = list(range(N_EXP))
fig, axes = plt.subplots(2, 2, figsize=(12, 7))

# Norma sin clip
norms_nc_finite = [n if np.isfinite(n) else np.nan for n in norms_nc]
axes[0,0].plot(steps, norms_nc_finite, color="crimson")
axes[0,0].set_title("Norma gradiente — sin clipping"); axes[0,0].set_xlabel("paso")
axes[0,0].set_ylabel("||g||"); axes[0,0].set_yscale("symlog", linthresh=10)

# Norma con clip
axes[0,1].plot(steps, norms_cl, color="steelblue")
axes[0,1].axhline(1.0, ls="--", color="gray", label="clip=1.0")
axes[0,1].set_title("Norma gradiente — con clipping"); axes[0,1].set_xlabel("paso")
axes[0,1].set_ylabel("||g||"); axes[0,1].legend()

# Loss sin clip
losses_nc_f = [l if np.isfinite(l) else np.nan for l in losses_nc]
axes[1,0].plot(steps, losses_nc_f, color="crimson")
axes[1,0].set_title("Loss — sin clipping"); axes[1,0].set_xlabel("paso")
axes[1,0].set_ylabel("CE loss")

# Loss con clip
axes[1,1].plot(steps, losses_cl, color="steelblue")
axes[1,1].set_title("Loss — con clipping"); axes[1,1].set_xlabel("paso")
axes[1,1].set_ylabel("CE loss")

plt.suptitle(f"Gradient Explosion (LR={HIGH_LR}) vs. Clipping", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()

pct_nan = sum(1 for n in norms_nc if not np.isfinite(n)) / N_EXP * 100
print(f"Sin clip: {pct_nan:.0f}% de pasos con norma NaN/inf")
print(f"Con clip: norma máxima = {max(norms_cl):.2f} (controlada)")


<a id="10-vanish"></a>
## 10. Vanishing Gradients y Pre-LayerNorm

### 10.1 El problema del vanishing

En redes profundas con **post-LN** (la norma *después* de la suma residual),
el gradiente puede desvanecerse a medida que retrocede por las capas:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}_0} = \prod_{l=1}^{L} \frac{\partial \mathbf{x}_l}{\partial \mathbf{x}_{l-1}}$$

Cada Jacobiano tiene valores singulares $< 1$ si las capas "comprimen"
la magnitud. Con $L=12$ capas, el gradiente puede caer a $10^{-6}$.

### 10.2 Cómo pre-LN + residuals mitigan el vanishing

**Conexión residual** (He et al., 2016):
$$\mathbf{x}_{l+1} = \mathbf{x}_l + F_l(\mathbf{x}_l)$$

El gradiente fluye **directamente** por la rama residual:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}_l} = \frac{\partial \mathcal{L}}{\partial \mathbf{x}_{l+1}} \left(1 + \frac{\partial F_l}{\partial \mathbf{x}_l}\right)$$

El término $+1$ garantiza que el gradiente siempre tenga una componente de
magnitud comparable a la capa superior, independientemente de $F_l$.

**Pre-LN** (antes de atención/FFN): normaliza *antes* de la transformación no
lineal → las activaciones entran a la atención con varianza controlada → los
Jacobianos son mejor condicionados. En post-LN, la norma se aplica *después*
de la suma residual, lo que puede hacer que la rama residual sea insignificante
al inicio.

### 10.3 Gradient Flow Plot

Para diagnosticar: medir la norma media del gradiente capa por capa.
Una curva plana = buen flujo. Exponencialmente decreciente = vanishing.


In [ ]:
# ── Gradient flow plot por capa ──────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# Usamos model_base entrenado en §5
x_gf, y_gf = next(iter(train_loader))
x_gf, y_gf = x_gf.to(DEVICE), y_gf.to(DEVICE)

model_base.train()
model_base.zero_grad()
import torch.nn.functional as F
logits_gf = model_base(x_gf)
loss_gf   = F.cross_entropy(logits_gf.view(-1, model_base.lm_head.out_features),
                              y_gf.view(-1))
loss_gf.backward()

# Recolectar norma de gradiente por capa nombrada
layer_names, grad_norms = [], []
for name, param in model_base.named_parameters():
    if param.grad is not None:
        norm = param.grad.norm(2).item()
        layer_names.append(name)
        grad_norms.append(norm)

# Acortar nombres para visualizar
short_names = [n.replace("blocks.", "blk").replace(".weight", ".W")
                .replace(".bias", ".b")[:30] for n in layer_names]

fig, ax = plt.subplots(figsize=(14, 5))
colors = ["steelblue" if "blocks" in n else "darkorange" for n in layer_names]
bars = ax.bar(range(len(grad_norms)), grad_norms, color=colors, alpha=0.8)
ax.set_xticks(range(len(short_names)))
ax.set_xticklabels(short_names, rotation=75, ha="right", fontsize=7)
ax.set_ylabel("Norma del gradiente ||∂L/∂W||")
ax.set_title("Gradient Flow — norma por capa (azul=bloques transformer, naranja=resto)")
ax.set_yscale("log")
plt.tight_layout(); plt.show()

print(f"Norma máxima: {max(grad_norms):.4f} | Norma mínima: {min(grad_norms):.6f}")
print(f"Ratio max/min: {max(grad_norms)/max(min(grad_norms),1e-9):.1f}x")
print("Una distribución uniforme indica buen flujo de gradiente (pre-LN + residuals).")


In [ ]:
# ── Pre-LN vs Post-LN: comparar normas por capa ──────────────────────────────
import torch
import torch.nn as nn
import copy

class TransformerBlockPostLN(nn.Module):
    """Bloque con Post-LayerNorm (menos estable): attn → norm, ffn → norm."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn  = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                            batch_first=True)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff1   = nn.Linear(d_model, d_ff)
        self.ff2   = nn.Linear(d_ff, d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x):
        # Post-LN: aplica norma DESPUÉS de sumar residual
        attn_mask = torch.triu(
            torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1
        ).bool()
        a, _ = self.attn(x, x, x, attn_mask=attn_mask, need_weights=False)
        x    = self.norm1(x + a)
        f    = self.drop(self.ff2(F.gelu(self.ff1(x))))
        x    = self.norm2(x + f)
        return x

def measure_grad_flow(model_variant, x, y, vocab_size):
    model_variant.train()
    model_variant.zero_grad()
    logits = model_variant(x)
    loss   = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    norms = []
    for p in model_variant.parameters():
        if p.grad is not None:
            norms.append(p.grad.norm(2).item())
    return norms

# Comparar MiniGPT (pre-LN) vs. versión con post-LN manual
# Usamos model_base (pre-LN ya entrenado)
norms_preln = measure_grad_flow(model_base, x_gf, y_gf, CFG["vocab_size"])

# Construir modelo equivalente post-LN (usando nn.TransformerDecoder simplificado)
# Para la demo usamos los mismos pesos iniciales pero distinta arquitectura
class SimplePostLNGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks  = nn.ModuleList([
            TransformerBlockPostLN(d_model, n_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx):
        B, T = idx.shape
        pos  = torch.arange(T, device=idx.device).unsqueeze(0)
        x    = self.tok_emb(idx) + self.pos_emb(pos)
        for blk in self.blocks:
            x = blk(x)
        return self.lm_head(x)

model_postln = SimplePostLNGPT(
    CFG["vocab_size"], CFG["d_model"], CFG["n_layers"],
    CFG["n_heads"], CFG["d_ff"], CFG["max_seq_len"]
).to(DEVICE)

norms_postln = measure_grad_flow(model_postln, x_gf, y_gf, CFG["vocab_size"])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(norms_preln,  label="Pre-LN (MiniGPT)", alpha=0.8, color="steelblue")
ax.plot(norms_postln, label="Post-LN",           alpha=0.8, color="crimson", ls="--")
ax.set_xlabel("Índice de parámetro"); ax.set_ylabel("Norma del gradiente")
ax.set_title("Gradient Flow: Pre-LN vs Post-LN (modelo sin entrenar)")
ax.set_yscale("log"); ax.legend()
plt.tight_layout(); plt.show()


<a id="11-ckpt"></a>
## 11. Checkpointing Completo y Reanudación Exacta

### 11.1 ¿Qué debe guardar un checkpoint?

Un checkpoint completo permite **reanudar exactamente** donde se paró:

```python
checkpoint = {
    "step":                int,
    "model_state_dict":    model.state_dict(),
    "optimizer_state_dict":optimizer.state_dict(),
    "scheduler_state_dict":scheduler.state_dict(),
    "scaler_state_dict":   scaler.state_dict(),    # GradScaler si fp16
    "rng_state": {
        "python": random.getstate(),
        "numpy":  np.random.get_state(),
        "torch":  torch.get_rng_state(),
        "cuda":   torch.cuda.get_rng_state_all(),  # si GPU
    },
    "config":   dict,   # hiperparámetros completos del run
    "metrics":  dict,   # última loss/val_loss/ppl
}
```

Si falta **cualquiera** de estos componentes:
- Sin estado del optimizador (momentos de Adam): los primeros pasos
  después de reanudar divergen.
- Sin estado del scheduler: la LR se reinicia (warmup se repite).
- Sin estado del RNG: el batch shuffle es diferente → curvas distintas.

### 11.2 Demostración de reanudación exacta

Entrenamos `100 steps → guardar → 100 steps más`.
Comparamos con un entrenamiento continuo de `200 steps` con la misma seed.
Las curvas deben ser **idénticas** (allclose).


In [ ]:
# ── Funciones save/load checkpoint ───────────────────────────────────────────
import random
from pathlib import Path

CKPT_DIR = Path("checkpoints/nb06_demo")
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(path, step, model, optimizer, scheduler, scaler, config, metrics):
    rng_state = {
        "python": random.getstate(),
        "numpy":  np.random.get_state(),
        "torch":  torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        rng_state["cuda"] = torch.cuda.get_rng_state_all()

    torch.save({
        "step":                 step,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict() if scheduler else None,
        "scaler_state_dict":    scaler.state_dict()    if scaler    else None,
        "rng_state":            rng_state,
        "config":               config,
        "metrics":              metrics,
    }, path)

def load_checkpoint(path, model, optimizer, scheduler=None, scaler=None):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if scheduler and ckpt["scheduler_state_dict"]:
        scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if scaler and ckpt["scaler_state_dict"]:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    # restaurar RNG
    rng = ckpt["rng_state"]
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"])
    if torch.cuda.is_available() and "cuda" in rng:
        torch.cuda.set_rng_state_all(rng["cuda"])
    return ckpt["step"], ckpt.get("metrics", {})

print("Funciones save/load definidas.")
print(f"Directorio de checkpoints: {CKPT_DIR}")


In [ ]:
# ── Demo: 100 steps → guardar → continuar 100 más vs 200 continuos ───────────
import itertools, math, torch, copy
import torch.nn.functional as F

DEMO_STEPS = 150   # total de pasos a comparar
HALF       = DEMO_STEPS // 2

def make_fresh_model_and_opt(seed=SEED):
    set_seed(seed)
    m = MiniGPT(
        vocab_size  = CFG["vocab_size"],
        d_model     = CFG["d_model"],
        n_layers    = CFG["n_layers"],
        n_heads     = CFG["n_heads"],
        d_ff        = CFG["d_ff"],
        max_seq_len = CFG["max_seq_len"],
        dropout     = 0.0,
    ).to(DEVICE)
    o = torch.optim.AdamW(m.parameters(), lr=CFG["lr"],
                           weight_decay=CFG["weight_decay"], betas=(0.9,0.95))
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=DEMO_STEPS)
    return m, o, sched

# ── Entrenamiento A: 200 pasos continuos ─────────────────────────────────────
model_A, opt_A, sched_A = make_fresh_model_and_opt(SEED)
set_seed(SEED)
iter_A = itertools.cycle(DataLoader(train_ds, batch_size=16,
                                     shuffle=True, drop_last=True))
losses_A = []
for step in range(DEMO_STEPS):
    x, y = next(iter_A)
    x, y = x.to(DEVICE), y.to(DEVICE)
    opt_A.zero_grad()
    logits = model_A(x)
    loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
    loss.backward()
    opt_A.step()
    sched_A.step()
    losses_A.append(loss.item())

# ── Entrenamiento B: HALF pasos → checkpoint → HALF más ──────────────────────
model_B, opt_B, sched_B = make_fresh_model_and_opt(SEED)
set_seed(SEED)
iter_B = itertools.cycle(DataLoader(train_ds, batch_size=16,
                                     shuffle=True, drop_last=True))
losses_B = []

# Primera mitad
for step in range(HALF):
    x, y = next(iter_B)
    x, y = x.to(DEVICE), y.to(DEVICE)
    opt_B.zero_grad()
    logits = model_B(x)
    loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
    loss.backward()
    opt_B.step()
    sched_B.step()
    losses_B.append(loss.item())

# Guardar checkpoint
ckpt_path = CKPT_DIR / "step_half.pt"
save_checkpoint(ckpt_path, HALF, model_B, opt_B, sched_B, None,
                config=CFG, metrics={"loss": losses_B[-1]})
print(f"Checkpoint guardado en {ckpt_path}  ({ckpt_path.stat().st_size/1024:.1f} KB)")

# Borrar modelo y cargar desde checkpoint
del model_B, opt_B, sched_B
model_B, opt_B, sched_B = make_fresh_model_and_opt(SEED + 99)  # seed diferente
step_resumed, _ = load_checkpoint(ckpt_path, model_B, opt_B, sched_B)
print(f"Checkpoint cargado — reanudando desde step {step_resumed}")

# Segunda mitad
for step in range(HALF, DEMO_STEPS):
    x, y = next(iter_B)
    x, y = x.to(DEVICE), y.to(DEVICE)
    opt_B.zero_grad()
    logits = model_B(x)
    loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
    loss.backward()
    opt_B.step()
    sched_B.step()
    losses_B.append(loss.item())

# ── Comparar curvas ───────────────────────────────────────────────────────────
diff = [abs(a - b) for a, b in zip(losses_A, losses_B)]
max_diff = max(diff)
print(f"\nDiferencia máxima entre curvas: {max_diff:.2e}")
curves_match = max_diff < 1e-4
print(f"Curvas idénticas (diff < 1e-4): {curves_match}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
steps_x = list(range(DEMO_STEPS))
axes[0].plot(steps_x, losses_A, label="Continuo (A)", alpha=0.8)
axes[0].plot(steps_x, losses_B, label="Con checkpoint (B)", alpha=0.8, ls="--")
axes[0].axvline(HALF, ls=":", color="gray", label=f"checkpoint @ step {HALF}")
axes[0].set_title("Loss: continuo vs. reanudado"); axes[0].legend()
axes[0].set_xlabel("paso"); axes[0].set_ylabel("loss")

axes[1].plot(steps_x, diff, color="green")
axes[1].axhline(1e-4, ls="--", color="gray", label="umbral 1e-4")
axes[1].set_title("Diferencia absoluta |loss_A - loss_B|")
axes[1].set_xlabel("paso"); axes[1].set_ylabel("|ΔL|"); axes[1].legend()
axes[1].set_yscale("log")

plt.suptitle("Reanudación exacta con checkpointing", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


In [ ]:
# ── Guardar en Google Drive si estamos en Colab ───────────────────────────────
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_ckpt_dir = Path("/content/drive/MyDrive/llm_lab_checkpoints")
        drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy(ckpt_path, drive_ckpt_dir / ckpt_path.name)
        print(f"Checkpoint copiado a Google Drive: {drive_ckpt_dir / ckpt_path.name}")
    except Exception as e:
        print(f"No se pudo montar Drive: {e}")
        print("Checkpoint disponible localmente en:", ckpt_path)
else:
    print(f"Checkpoint local: {ckpt_path}")
    print("En Colab: se montaría Google Drive y se copiaría automáticamente.")


<a id="12-lr"></a>
## 12. Learning Rate Schedulers: Warmup + Cosine Decay

### 12.1 ¿Por qué warmup?

Al inicio del entrenamiento:
- Los pesos son aleatorios → los gradientes son grandes e inestables.
- Adam aún no tiene buenas estimaciones de varianza ($v_t \approx 0$ → paso grande).
- Un LR alto en estos pasos iniciales puede destruir los pesos antes de
  que el modelo encuentre una región prometedora.

**Warmup lineal**: aumentar la LR linealmente desde 0 hasta `lr_max`
en los primeros `warmup_steps` pasos. Esto da tiempo a Adam para calibrar
sus estadísticas.

### 12.2 Cosine Decay (derivación)

Después del warmup, la LR sigue un coseno:

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_{\max} - \alpha_{\min}) \left(1 + \cos\!\left(\pi \frac{t - t_w}{T - t_w}\right)\right)$$

donde:
- $t_w$ = `warmup_steps`
- $T$ = `max_steps`
- $\alpha_{\min}$ = LR final (típicamente $0.1 \times \alpha_{\max}$)

**Intuición**: el coseno decae suavemente desde $\alpha_{\max}$ hasta $\alpha_{\min}$,
acelerando el descenso al inicio y desacelerándolo al final (cuando el modelo
está cerca de un mínimo y queremos pasos finos).


In [ ]:
# ── Implementación manual de warmup lineal + cosine decay ────────────────────
import math

def lr_warmup_cosine(step: int, warmup_steps: int, max_steps: int,
                      lr_max: float, lr_min: float = 0.0) -> float:
    """LR schedule: rampa lineal hasta warmup_steps, luego coseno hasta max_steps."""
    if step < warmup_steps:
        # Warmup lineal: 0 → lr_max
        return lr_max * step / max(warmup_steps, 1)
    elif step >= max_steps:
        return lr_min
    else:
        # Cosine decay: lr_max → lr_min
        progress = (step - warmup_steps) / max(max_steps - warmup_steps, 1)
        return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * progress))

# Visualizar distintos schedulers
TOTAL   = 1000
WARMUP  = 100
LR_MAX  = 3e-4
LR_MIN  = LR_MAX * 0.1
steps_v = list(range(TOTAL))

lr_cosine   = [lr_warmup_cosine(s, WARMUP, TOTAL, LR_MAX, LR_MIN) for s in steps_v]
lr_constant = [LR_MAX if s >= WARMUP else LR_MAX * s / WARMUP for s in steps_v]
lr_linear   = [max(LR_MIN, LR_MAX * (1 - (s - WARMUP) / max(TOTAL - WARMUP, 1)))
               if s >= WARMUP else LR_MAX * s / WARMUP for s in steps_v]
lr_step     = [LR_MAX * (0.5 ** (s // 300)) if s >= WARMUP
               else LR_MAX * s / WARMUP for s in steps_v]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps_v, lr_cosine,   label="warmup + cosine (recomendado)", lw=2)
ax.plot(steps_v, lr_constant, label="warmup + constant",             lw=1.5, ls="--")
ax.plot(steps_v, lr_linear,   label="warmup + linear decay",         lw=1.5, ls="-.")
ax.plot(steps_v, lr_step,     label="warmup + step decay",           lw=1.5, ls=":")
ax.axvline(WARMUP, color="gray", ls=":", alpha=0.7, label=f"warmup_end={WARMUP}")
ax.set_xlabel("paso"); ax.set_ylabel("learning rate")
ax.set_title("Comparación de LR Schedulers (warmup lineal + distintos decays)")
ax.legend(fontsize=9); ax.set_yscale("log")
plt.tight_layout(); plt.show()

# Verificar que nuestra implementación coincide con PyTorch CosineAnnealingLR
import torch
dummy_model = torch.nn.Linear(2, 2)
dummy_opt   = torch.optim.SGD(dummy_model.parameters(), lr=LR_MAX)
# No hay warmup en CosineAnnealingLR puro, así que comparamos solo la fase cosine
sched_torch = torch.optim.lr_scheduler.CosineAnnealingLR(
    dummy_opt, T_max=TOTAL - WARMUP, eta_min=LR_MIN)
lrs_torch = []
for _ in range(TOTAL - WARMUP):
    lrs_torch.append(dummy_opt.param_groups[0]["lr"])
    sched_torch.step()

our_cosine  = [lr_warmup_cosine(s, 0, TOTAL - WARMUP, LR_MAX, LR_MIN)
               for s in range(TOTAL - WARMUP)]
max_diff    = max(abs(a - b) for a, b in zip(our_cosine, lrs_torch))
print(f"Diferencia máxima vs PyTorch CosineAnnealingLR: {max_diff:.2e}")


In [ ]:
# ── Comparar schedulers con runs cortos ──────────────────────────────────────
import itertools

def quick_train(lr_fn, n_steps=200, batch_sz=16, seed=SEED):
    """Entrena n_steps pasos con un schedule dado; devuelve lista de losses."""
    set_seed(seed)
    m = MiniGPT(
        vocab_size=CFG["vocab_size"], d_model=64, n_layers=2,
        n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
    ).to(DEVICE)
    o = torch.optim.AdamW(m.parameters(), lr=1.0, betas=(0.9, 0.95))
    loader_q = DataLoader(train_ds, batch_size=batch_sz, shuffle=True,
                          drop_last=True, num_workers=0)
    it = itertools.cycle(loader_q)
    losses = []
    for step in range(n_steps):
        lr = lr_fn(step)
        for pg in o.param_groups:
            pg["lr"] = lr
        x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        o.zero_grad()
        logits = m(x)
        loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        o.step()
        losses.append(loss.item())
    return losses

N_COMPARE = 300
WSTEPS    = 30

schedulers = {
    "cosine":    lambda s: lr_warmup_cosine(s, WSTEPS, N_COMPARE, LR_MAX, LR_MIN),
    "constant":  lambda s: LR_MAX if s >= WSTEPS else LR_MAX * s / WSTEPS,
    "linear":    lambda s: max(LR_MIN, LR_MAX * (1 - (s-WSTEPS)/max(N_COMPARE-WSTEPS,1)))
                           if s >= WSTEPS else LR_MAX * s / WSTEPS,
    "step":      lambda s: LR_MAX * (0.5 ** (s // 100)) if s >= WSTEPS
                           else LR_MAX * s / WSTEPS,
}

print("Entrenando runs cortos para comparar schedulers...")
results_sched = {}
for name, fn in schedulers.items():
    losses_run = quick_train(fn, n_steps=N_COMPARE)
    results_sched[name] = losses_run
    print(f"  {name}: loss_final={losses_run[-1]:.4f}")

# Smoothing para visualizar
def smooth(arr, w=15):
    return [sum(arr[max(0,i-w):i+1]) / len(arr[max(0,i-w):i+1]) for i in range(len(arr))]

fig, ax = plt.subplots(figsize=(10, 5))
for name, losses in results_sched.items():
    ax.plot(smooth(losses, 20), label=name, lw=2)
ax.set_xlabel("paso"); ax.set_ylabel("loss (suavizada)")
ax.set_title("Comparación de schedulers — misma arquitectura, misma seed")
ax.legend(); plt.tight_layout(); plt.show()


<a id="13-optim"></a>
## 13. Optimizadores: AdamW, SGD+momentum, Adafactor, Lion

### 13.1 AdamW

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\hat{m}_t = m_t / (1-\beta_1^t), \quad \hat{v}_t = v_t / (1-\beta_2^t)$$
$$\theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon} - \alpha\lambda\theta_{t-1}$$

El término $-\alpha\lambda\theta_{t-1}$ es el **weight decay desacoplado** (Loshchilov & Hutter, 2019).

### 13.2 SGD + Momentum

$$v_t = \mu v_{t-1} + g_t$$
$$\theta_t = \theta_{t-1} - \alpha v_t$$

Más simple, requiere tuning cuidadoso de LR. Generalmente peor que Adam en LLMs.

### 13.3 Adafactor (Shazeer & Stern, 2018)

Factoriza la varianza de segundo momento $V \approx R \cdot C^T$ (rank-1):

$$R_t = \beta_2^r R_{t-1} + (1-\beta_2^r) \|g_t\|_{\text{filas}}^2$$
$$C_t = \beta_2^c C_{t-1} + (1-\beta_2^c) \|g_t\|_{\text{columnas}}^2$$
$$\hat{v}_{ij} \approx \frac{R_{t,i} \cdot C_{t,j}}{\sum_k R_{t,k}}$$

Esto reduce la memoria del estado del optimizador de $O(n \cdot m)$ a $O(n + m)$
— crucial para modelos grandes.

### 13.4 Lion (Chen et al., 2023)

Lion solo mantiene el **signo** del gradiente actualizado:

$$c_t = \text{sign}(\beta_1 m_{t-1} + (1-\beta_1) g_t)$$
$$\theta_t = \theta_{t-1} - \alpha (c_t + \lambda\theta_{t-1})$$
$$m_t = \beta_2 m_{t-1} + (1-\beta_2) g_t$$

Usa **menos memoria** que Adam (solo un momento en lugar de dos) y
ha mostrado mejoras en vision transformers y algunos LLMs.


In [ ]:
# ── Adafactor: implementación desde cero (factorización rank-1) ──────────────
import torch
from torch.optim import Optimizer

class Adafactor(Optimizer):
    """Adafactor simplificado con factorización rank-1 del segundo momento.

    Referencia: Shazeer & Stern (2018), https://arxiv.org/abs/1805.09843
    """
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-30,
                 weight_decay=0.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eps=eps,
                        weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                g   = p.grad
                lr  = group["lr"]
                b1  = group["beta1"]
                b2  = group["beta2"]
                eps = group["eps"]
                wd  = group["weight_decay"]
                st  = self.state[p]

                if len(st) == 0:
                    st["step"] = 0
                    st["m"]    = torch.zeros_like(p)   # primer momento
                    if g.dim() >= 2:
                        # Factorización rank-1: R (filas) y C (columnas)
                        st["R"]  = torch.zeros(g.shape[0], device=p.device)
                        st["C"]  = torch.zeros(g.shape[1], device=p.device)
                    else:
                        st["v"] = torch.zeros_like(p)

                st["step"] += 1
                t  = st["step"]

                # Corrección de bias
                bc1 = 1 - b1 ** t
                bc2 = 1 - b2 ** t

                # Actualizar primer momento
                st["m"].mul_(b1).add_(g, alpha=1 - b1)
                m_hat = st["m"] / bc1

                # Segundo momento (factorizado o escalar)
                if g.dim() >= 2:
                    g2     = g ** 2 + eps
                    r_new  = g2.mean(dim=1)   # norma cuadrada por fila
                    c_new  = g2.mean(dim=0)   # norma cuadrada por columna
                    st["R"].mul_(b2).add_(r_new, alpha=1 - b2)
                    st["C"].mul_(b2).add_(c_new, alpha=1 - b2)
                    R_hat  = st["R"] / bc2
                    C_hat  = st["C"] / bc2
                    # Reconstruir v aproximado (rank-1)
                    v_hat  = torch.outer(R_hat, C_hat) / (R_hat.sum() + eps)
                else:
                    st["v"].mul_(b2).add_(g ** 2, alpha=1 - b2)
                    v_hat = st["v"] / bc2

                # Paso del optimizador
                update = m_hat / (v_hat.sqrt() + eps)
                p.add_(update, alpha=-lr)
                if wd > 0:
                    p.add_(p, alpha=-lr * wd)

        return loss

print("Adafactor implementado correctamente.")
# Verificación rápida
test_model = torch.nn.Linear(4, 4)
test_opt   = Adafactor(test_model.parameters(), lr=1e-3)
x_test     = torch.randn(8, 4)
loss_test  = test_model(x_test).sum()
loss_test.backward()
test_opt.step()
print("Adafactor.step() ejecutado sin errores.")


In [ ]:
# ── Lion: implementación desde cero (~15 líneas) ─────────────────────────────
class Lion(Optimizer):
    """Lion optimizer (Chen et al., 2023): https://arxiv.org/abs/2302.06675

    Solo usa el signo del gradiente actualizado — menos memoria que Adam.
    """
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None if closure is None else closure()
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                g   = p.grad
                b1, b2 = group["betas"]
                lr  = group["lr"]
                wd  = group["weight_decay"]
                st  = self.state[p]
                if "m" not in st:
                    st["m"] = torch.zeros_like(p)

                m = st["m"]
                # Calcular dirección de actualización: sign(β1*m + (1-β1)*g)
                update = (b1 * m + (1 - b1) * g).sign()
                # Actualizar parámetros
                p.add_(update + wd * p, alpha=-lr)
                # Actualizar momento con β2
                m.mul_(b2).add_(g, alpha=1 - b2)
        return loss

print("Lion implementado correctamente.")
test_opt_lion = Lion(test_model.parameters(), lr=1e-4)
test_model.zero_grad()
loss_test2 = test_model(x_test).sum()
loss_test2.backward()
test_opt_lion.step()
print("Lion.step() ejecutado sin errores.")


In [ ]:
# ── Comparar 4 optimizadores: AdamW, SGD, Adafactor, Lion ────────────────────
import itertools, time
import torch.nn.functional as F

OPTIM_STEPS = 300

optimizers_cfg = {
    "AdamW": lambda m: torch.optim.AdamW(m.parameters(), lr=3e-4,
                                          weight_decay=0.1, betas=(0.9, 0.95)),
    "SGD+mom": lambda m: torch.optim.SGD(m.parameters(), lr=3e-3, momentum=0.9,
                                          weight_decay=0.01, nesterov=True),
    "Adafactor": lambda m: Adafactor(m.parameters(), lr=3e-4, weight_decay=0.01),
    "Lion":      lambda m: Lion(m.parameters(), lr=1e-4, weight_decay=0.01),
}

results_optim = {}
mem_per_param  = {}

for opt_name, opt_fn in optimizers_cfg.items():
    set_seed(SEED)
    m = MiniGPT(
        vocab_size=CFG["vocab_size"], d_model=64, n_layers=2,
        n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
    ).to(DEVICE)
    o = opt_fn(m)
    loader_o = DataLoader(train_ds, batch_size=16, shuffle=True,
                          drop_last=True, num_workers=0)
    it       = itertools.cycle(loader_o)
    losses_o = []
    for step in range(OPTIM_STEPS):
        x, y = next(it)
        x, y = x.to(DEVICE), y.to(DEVICE)
        o.zero_grad()
        logits = m(x)
        loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        o.step()
        losses_o.append(loss.item())

    # Memoria del estado del optimizador en bytes/param
    n_p   = sum(p.numel() for p in m.parameters())
    total_state_bytes = 0
    for s_dict in o.state.values():
        for v in s_dict.values():
            if isinstance(v, torch.Tensor):
                total_state_bytes += v.numel() * v.element_size()
    bytes_per_param = total_state_bytes / max(n_p, 1)
    mem_per_param[opt_name] = bytes_per_param

    results_optim[opt_name] = losses_o
    print(f"{opt_name:<12} loss_final={losses_o[-1]:.4f}  "
          f"estado={bytes_per_param:.1f} bytes/param")

# ── Gráficas ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for opt_name, losses_o in results_optim.items():
    axes[0].plot(smooth(losses_o, 20), label=opt_name, lw=2)
axes[0].set_xlabel("paso"); axes[0].set_ylabel("loss (suavizada)")
axes[0].set_title("Loss por optimizador"); axes[0].legend()

axes[1].bar(list(mem_per_param.keys()), list(mem_per_param.values()),
             color=["steelblue", "darkorange", "green", "crimson"])
axes[1].set_ylabel("Bytes por parámetro")
axes[1].set_title("Memoria del estado del optimizador")
for i, (k, v) in enumerate(mem_per_param.items()):
    axes[1].text(i, v + 0.05, f"{v:.1f}", ha="center", fontsize=10)

plt.suptitle("Comparación de Optimizadores", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()


<a id="14-curriculum"></a>
## 14. Curriculum Learning

### 14.1 Concepto

**Curriculum learning** (Bengio et al., 2009) ordena los ejemplos de
entrenamiento de más fácil a más difícil, imitando el aprendizaje humano.

**Intuición**: al inicio del entrenamiento, los ejemplos difíciles
generan gradientes muy ruidosos que no aportan señal útil. Empezar
con ejemplos fáciles establece una base sólida antes de exponer el
modelo a casos complejos.

### 14.2 Medidas de dificultad

Para un modelo de lenguaje, la dificultad de una secuencia se puede medir con:
- **Perplexity de un modelo de referencia** (más alta = más difícil).
- **Longitud de la secuencia** (proxy simple).
- **Entropía del texto** (alta entropía = más predecible o más aleatorio?).
- **Loss del modelo actual** (difícil = pérdida alta en el momento actual —
  *online* curriculum).

### 14.3 Limitaciones

En pre-training moderno, el curriculum es menos importante porque:
1. El corpus es tan grande que el modelo ve todos los ejemplos muchas veces.
2. El ruido de gradient a escala es manejable con batch sizes grandes.
3. Implementar y mantener un curriculum a escala es costoso.

Se usa más en **fine-tuning** y **RLHF** donde el conjunto de datos es pequeño.


In [ ]:
# ── Mini-demo: curriculum (por longitud/dificultad) vs. aleatorio ─────────────
import itertools, torch
import torch.nn.functional as F

# Crear dataset con ejemplos de distintas "dificultades" (medida: entropía del texto)
def text_difficulty(text: str) -> float:
    """Entropía de caracteres como proxy de dificultad."""
    from collections import Counter
    import math
    if not text:
        return 0.0
    counts = Counter(text)
    total  = len(text)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

# Calcular dificultad de cada chunk del dataset
chunk_difficulties = []
for i in range(min(len(train_ds), 200)):
    ids   = train_ds[i][0].tolist()
    text  = tokenizer.decode(ids)
    diff  = text_difficulty(text)
    chunk_difficulties.append((i, diff))

# Ordenar de fácil a difícil
sorted_by_diff    = sorted(chunk_difficulties, key=lambda x: x[1])
easy_to_hard_idx  = [x[0] for x in sorted_by_diff]
random_idx        = list(range(len(chunk_difficulties)))
random.shuffle(random_idx)

def train_with_order(indices, n_steps=200, seed=SEED):
    """Entrena sobre chunks en el orden dado."""
    set_seed(seed)
    m = MiniGPT(
        vocab_size=CFG["vocab_size"], d_model=64, n_layers=2,
        n_heads=2, d_ff=256, max_seq_len=CFG["max_seq_len"], dropout=0.0
    ).to(DEVICE)
    o = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=0.1)
    losses = []
    for step, idx in enumerate(itertools.cycle(indices)):
        if step >= n_steps:
            break
        x, y = train_ds[idx]
        x    = x.unsqueeze(0).to(DEVICE)
        y    = y.unsqueeze(0).to(DEVICE)
        o.zero_grad()
        logits = m(x)
        loss   = F.cross_entropy(logits.view(-1, CFG["vocab_size"]), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0)
        o.step()
        losses.append(loss.item())
    return losses

print("Entrenando con curriculum (fácil → difícil)...")
losses_curr   = train_with_order(easy_to_hard_idx)
print("Entrenando con orden aleatorio...")
losses_random = train_with_order(random_idx)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(smooth(losses_curr,   20), label="Curriculum (fácil→difícil)", lw=2, color="steelblue")
ax.plot(smooth(losses_random, 20), label="Aleatorio",                   lw=2, color="crimson", ls="--")
ax.set_xlabel("paso"); ax.set_ylabel("loss (suavizada)")
ax.set_title("Curriculum Learning vs. Orden Aleatorio")
ax.legend(); plt.tight_layout(); plt.show()

diff_vals = [x[1] for x in sorted_by_diff]
print(f"\nRango de dificultad (entropía): {min(diff_vals):.2f} – {max(diff_vals):.2f} bits/char")
print(f"Loss final curriculum: {losses_curr[-1]:.4f}  |  random: {losses_random[-1]:.4f}")
